In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch
import torch.nn as nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# --- 1. Load QM9 (downloads automatically, ~130k molecules) ---
dataset = QM9(root='/kaggle/working/qm9_data')
print(dataset)
print("One molecule:", dataset[0])

# gap is target index 4 in QM9's 19-column target tensor
TARGET_IDX = 4

# --- 2. Split into train/val/test (index-based, same concept as train_test_split) ---
dataset = dataset.shuffle()
n = len(dataset)
train_dataset = dataset[:int(0.8 * n)]
val_dataset   = dataset[int(0.8 * n):int(0.9 * n)]
test_dataset  = dataset[int(0.9 * n):]

print(f"train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

# --- 3. DataLoaders (this is the "batching" concept from earlier) ---
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)

# --- 4. Baseline: always predict the training set's mean gap ---
train_mean_gap = train_dataset.data.y[:, TARGET_IDX].mean().item()
val_targets = val_dataset.data.y[:, TARGET_IDX]
baseline_mae = (val_targets - train_mean_gap).abs().mean().item()
print("Baseline MAE (always predict mean):", baseline_mae)

# --- 5. Define the GNN ---
class GNN(nn.Module):
    def __init__(self, num_node_features, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index).relu()
        x = global_mean_pool(x, batch)   # pool atom embeddings -> one vector per molecule
        return self.linear(x)

model = GNN(num_node_features=dataset.num_node_features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.L1Loss()  # L1 loss = Mean Absolute Error, standard for QM9

# --- 6. Training loop ---
def train_one_epoch():
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch).squeeze()
        target = batch.y[:, TARGET_IDX]
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(train_dataset)

def evaluate(loader):
    model.eval()
    total_error = 0
    with torch.no_grad():
        for batch in loader:
            pred = model(batch.x, batch.edge_index, batch.batch).squeeze()
            target = batch.y[:, TARGET_IDX]
            total_error += (pred - target).abs().sum().item()
    return total_error / len(loader.dataset)

for epoch in range(30):
    train_loss = train_one_epoch()
    val_mae = evaluate(val_loader)
    print(f"epoch {epoch}: train loss {train_loss:.4f}, val MAE {val_mae:.4f}")

Extracting /kaggle/working/qm9_data/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


QM9(130831)
One molecule: Data(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name='gdb_1', z=[5])
train: 104664, val: 13083, test: 13084
Baseline MAE (always predict mean): 1.0741896629333496


/tmp/ipykernel_58/57762401.py:53: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The given 'InMemoryDataset' only references a subset of examples of the full dataset, but 'data' will contain information of the full dataset. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  train_mean_gap = train_dataset.data.y[:, TARGET_IDX].mean().item()
/tmp/ipykernel_58/57762401.py:54: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The given 'InMemoryDataset' only references a subset

epoch 0: train loss 0.9990, val MAE 0.7989
epoch 1: train loss 0.7882, val MAE 0.7837
epoch 2: train loss 0.7702, val MAE 0.7527
epoch 3: train loss 0.7560, val MAE 0.7480
epoch 4: train loss 0.7406, val MAE 0.7360
epoch 5: train loss 0.7170, val MAE 0.6942
epoch 6: train loss 0.6870, val MAE 0.6978
epoch 7: train loss 0.6581, val MAE 0.6449
epoch 8: train loss 0.6335, val MAE 0.6660
epoch 9: train loss 0.6199, val MAE 0.6160
epoch 10: train loss 0.6072, val MAE 0.6101
epoch 11: train loss 0.5968, val MAE 0.6019
epoch 12: train loss 0.5900, val MAE 0.5767
epoch 13: train loss 0.5822, val MAE 0.5718
epoch 14: train loss 0.5737, val MAE 0.5799
epoch 15: train loss 0.5666, val MAE 0.5643
epoch 16: train loss 0.5581, val MAE 0.5489
epoch 17: train loss 0.5516, val MAE 0.5424
epoch 18: train loss 0.5423, val MAE 0.5317
epoch 19: train loss 0.5372, val MAE 0.5212
epoch 20: train loss 0.5283, val MAE 0.5288
epoch 21: train loss 0.5220, val MAE 0.5347
epoch 22: train loss 0.5190, val MAE 0.514